# PAN experiment verification notebook

This notebook rebuilds the experiment stack from `pan.py` in notebook form so you can verify each experimental mode and compare the results against the README claims.

It includes notebook versions of these modes:

- `compare`
- `sweep`
- `primes`
- `tier3`
- `dw_sweep`
- `wd_sweep`
- `k8_sweep`
- `tf_sweep`
- `held_out_primes`

The cells are arranged so nothing heavy runs automatically. You turn on the experiment you want to execute.


## Recommended workflow

Use this notebook in three passes.

First, run the shared definitions and helper cells.  
Second, run one experiment section at a time.  
Third, compare your outputs against the expected patterns in the README rather than expecting bit-for-bit identical numbers on every device.

That matters because training trajectories can move around with seed, device, compile mode, and optimizer numerics.


In [ ]:
import math
import os
import time
import json
import warnings
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

PHASE_SCALE = 65536
PHASE_SCALE_F = 65536.0
TWO_PI = 2.0 * math.pi

print("device:", DEVICE)


## Core implementation

This section mirrors your current code closely so the notebook is a faithful verification environment rather than a different experimental framework.


In [ ]:
def _maybe_compile(model: nn.Module, use_compile: bool) -> nn.Module:
    if not use_compile:
        return model
    if not hasattr(torch, "compile"):
        warnings.warn("torch.compile not available; running in eager mode.")
        return model
    try:
        if hasattr(torch, "_dynamo"):
            torch._dynamo.config.suppress_errors = True
            torch._dynamo.config.recompile_limit = 64
        return torch.compile(model, backend="aot_eager")
    except Exception as e:
        warnings.warn(f"torch.compile failed ({e}); running in eager mode.")
        return model


def make_modular_dataset(p: int, train_frac: float = 0.4, seed: int = 42):
    rng = np.random.default_rng(seed)
    pairs = np.array([(a, b, (a + b) % p) for a in range(p) for b in range(p)], dtype=np.int64)
    perm = rng.permutation(len(pairs))
    pairs = pairs[perm]

    n_train = int(train_frac * len(pairs))
    train = torch.tensor(pairs[:n_train], device=DEVICE)
    val = torch.tensor(pairs[n_train:], device=DEVICE)

    return train[:, :2], train[:, 2], val[:, :2], val[:, 2]


class PhaseEncoder(nn.Module):
    def __init__(self, p: int, k_freqs: int):
        super().__init__()
        self.p = p
        self.k_freqs = k_freqs
        init_freqs = torch.tensor(
            [(k + 1) * TWO_PI / p for k in range(k_freqs)],
            dtype=torch.float32,
        )
        self.freq = nn.Parameter(init_freqs)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        a = tokens.float().unsqueeze(-1)
        f = self.freq.unsqueeze(0)
        phases = (a * f) % TWO_PI
        return phases


class PhaseMixingLayer(nn.Module):
    def __init__(self, n_in: int, n_out: int):
        super().__init__()
        w_init = torch.randn(n_out, n_in) * 0.1 + (1.0 / n_in)
        self.weight = nn.Parameter(w_init)

    def forward(self, phases_in: torch.Tensor) -> torch.Tensor:
        return F.linear(phases_in, self.weight) % TWO_PI


class PhaseGate(nn.Module):
    def __init__(self, n_phases: int):
        super().__init__()
        self.ref_phase = nn.Parameter(torch.rand(n_phases) * TWO_PI)

    def forward(self, phases: torch.Tensor) -> torch.Tensor:
        ref = torch.remainder(self.ref_phase, TWO_PI)
        phase_diff = phases - ref.unsqueeze(0)
        return (1.0 + torch.cos(phase_diff)) / 2.0


class PhaseAccumulatorNetwork(nn.Module):
    def __init__(self, p: int, k_freqs: int = 5):
        super().__init__()
        self.p = p
        self.k_freqs = k_freqs

        self.encoder_a = PhaseEncoder(p, k_freqs)
        self.encoder_b = PhaseEncoder(p, k_freqs)
        self.phase_mix = PhaseMixingLayer(2 * k_freqs, k_freqs)
        self.phase_gate = PhaseGate(k_freqs)
        self.decoder = nn.Linear(k_freqs, p, bias=True)

        nn.init.normal_(self.decoder.weight, std=0.02)
        nn.init.zeros_(self.decoder.bias)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        a = inputs[:, 0]
        b = inputs[:, 1]

        phi_a = self.encoder_a(a)
        phi_b = self.encoder_b(b)
        phi_mixed = self.phase_mix(torch.cat([phi_a, phi_b], dim=-1))
        gates = self.phase_gate(phi_mixed)
        return self.decoder(gates)

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def get_learned_frequencies(self) -> dict:
        freqs_a_raw = self.encoder_a.freq.detach().cpu().numpy()
        freqs_b_raw = self.encoder_b.freq.detach().cpu().numpy()
        freqs_a = freqs_a_raw % TWO_PI
        freqs_b = freqs_b_raw % TWO_PI
        theoretical = np.array([(k + 1) * TWO_PI / self.p for k in range(self.k_freqs)])

        def _angle_err(learned, theory):
            diff = np.abs(learned - theory) % TWO_PI
            return np.minimum(diff, TWO_PI - diff)

        return {
            "learned_a": freqs_a,
            "learned_b": freqs_b,
            "learned_a_raw": freqs_a_raw,
            "learned_b_raw": freqs_b_raw,
            "theoretical": theoretical,
            "error_a": _angle_err(freqs_a, theoretical),
            "error_b": _angle_err(freqs_b, theoretical),
        }


class TransformerBaseline(nn.Module):
    def __init__(self, p: int, d_model: int = 128, n_heads: int = 4, d_mlp: int = 512):
        super().__init__()
        self.p = p
        self.d_model = d_model

        self.tok_embed = nn.Embedding(p + 1, d_model)
        self.pos_embed = nn.Embedding(3, d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_mlp),
            nn.ReLU(),
            nn.Linear(d_mlp, d_model),
        )
        self.unembed = nn.Linear(d_model, p, bias=False)

        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, std=0.02)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        batch = inputs.shape[0]
        eq_token = torch.full((batch, 1), self.p, dtype=torch.long, device=inputs.device)
        seq = torch.cat([inputs, eq_token], dim=1)
        positions = torch.arange(3, device=inputs.device).unsqueeze(0)
        x = self.tok_embed(seq) + self.pos_embed(positions)
        mask = torch.triu(torch.ones(3, 3, device=inputs.device), diagonal=1).bool()
        x_attn, _ = self.attn(x, x, x, attn_mask=mask)
        x = x + x_attn
        x = x + self.mlp(x)
        return self.unembed(x[:, -1, :])

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


In [ ]:
@dataclass
class TrainConfig:
    p: int = 113
    n_steps: int = 50_000
    batch_size: int = 256
    lr: float = 1e-3
    weight_decay: float = 1.0
    log_every: int = 200
    seed: int = 42
    k_freqs: int = 5
    diversity_weight: float = 0.01
    d_model: int = 128
    n_heads: int = 4
    d_mlp: int = 512
    val_samples: Optional[int] = None
    use_compile: bool = True
    early_stop: bool = True
    output_dir: str = "."
    save_model: bool = False
    dry_run: bool = False


@dataclass
class TrainHistory:
    steps: list = field(default_factory=list)
    train_loss: list = field(default_factory=list)
    val_loss: list = field(default_factory=list)
    val_acc: list = field(default_factory=list)
    grok_step: Optional[int] = None
    freq_checkpoints: dict = field(default_factory=dict)
    fourier_conc_steps: list = field(default_factory=list)
    fourier_conc_values: list = field(default_factory=list)


def _make_val_loader(val_x, val_y, val_samples: Optional[int]):
    if val_samples is None or val_samples >= len(val_x):
        return val_x, val_y
    idx = torch.randperm(len(val_x), device=val_x.device)[:val_samples]
    return val_x[idx], val_y[idx]


def _print_run_config(cfg: TrainConfig, label: str) -> None:
    print(f"[config:{label}] p={cfg.p} k={cfg.k_freqs} seed={cfg.seed} steps={cfg.n_steps:,}")
    print(f"               lr={cfg.lr} wd={cfg.weight_decay} dw={cfg.diversity_weight} batch={cfg.batch_size}")
    print(f"               val_samples={cfg.val_samples or 'full'} compile={cfg.use_compile} early_stop={cfg.early_stop}")


def fourier_concentration(embed_W: torch.Tensor, top_k: int = 10) -> float:
    W = embed_W.float()
    if W.dim() == 1:
        W = W.unsqueeze(-1)
    W_fft = torch.fft.fft(W, dim=0)
    energy = W_fft.abs() ** 2
    total = energy.sum().item()
    if total < 1e-10:
        return 0.0
    return energy.reshape(-1).topk(top_k).values.sum().item() / total


def train(model: nn.Module, cfg: TrainConfig, train_x, train_y, val_x, val_y,
          label: str = "model", record_checkpoints: bool = False) -> TrainHistory:
    _print_run_config(cfg, label)
    if cfg.dry_run:
        print(f"[dry-run] would train {cfg.n_steps:,} steps")
        return TrainHistory()

    torch.manual_seed(cfg.seed)
    model = _maybe_compile(model, cfg.use_compile)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
    )

    eval_x, eval_y = _make_val_loader(val_x, val_y, cfg.val_samples)
    history = TrainHistory()
    n_train = len(train_x)
    t0 = time.time()

    for step in range(cfg.n_steps):
        model.train()

        idx = torch.randperm(n_train, device=DEVICE)[:cfg.batch_size]
        x_batch = train_x[idx]
        y_batch = train_y[idx]

        logits = model(x_batch)
        loss = F.cross_entropy(logits, y_batch)

        if cfg.diversity_weight > 0 and hasattr(model, "phase_mix"):
            with torch.no_grad():
                phi_a = model.encoder_a(x_batch[:, 0])
                phi_b = model.encoder_b(x_batch[:, 1])

            mix_out = model.phase_mix(torch.cat([phi_a, phi_b], dim=-1))
            mix_norm = mix_out - mix_out.mean(0, keepdim=True)
            norms = mix_norm.norm(dim=0, keepdim=True).clamp(min=1e-6)
            mix_norm = mix_norm / norms
            gram = mix_norm.T @ mix_norm / mix_out.shape[0]
            eye = torch.eye(gram.shape[0], device=gram.device)
            div_loss = (gram - eye).pow(2).sum() / gram.shape[0]
            loss = loss + cfg.diversity_weight * div_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % cfg.log_every == 0:
            model.eval()
            with torch.no_grad():
                val_logits = model(eval_x)
                val_loss = F.cross_entropy(val_logits, eval_y).item()
                val_acc = (val_logits.argmax(-1) == eval_y).float().mean().item()

            history.steps.append(step)
            history.train_loss.append(loss.item())
            history.val_loss.append(val_loss)
            history.val_acc.append(val_acc)

            if record_checkpoints and hasattr(model, "get_learned_frequencies"):
                raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model
                history.freq_checkpoints[step] = raw_model.get_learned_frequencies()
                dec_w = raw_model.decoder.weight.detach().float()
                history.fourier_conc_steps.append(step)
                history.fourier_conc_values.append(
                    fourier_concentration(dec_w, top_k=min(10, dec_w.shape[0]))
                )

            if val_acc > 0.99 and history.grok_step is None:
                history.grok_step = step
                print(f"★ [{label}] GROKKED at step {step:,} | val_acc={val_acc:.3f} | elapsed={time.time()-t0:.1f}s")
                if cfg.early_stop:
                    break

            if step % (cfg.log_every * 5) == 0:
                print(f"[{label}] step={step:6d} train_loss={loss.item():.4f} val_acc={val_acc:.4f} elapsed={time.time()-t0:.1f}s")

    return history


In [ ]:
def analyze_pan(model: PhaseAccumulatorNetwork) -> dict:
    with torch.no_grad():
        freq_info = model.get_learned_frequencies()

    print("k | theoretical | learned_a | err_a | learned_b | err_b")
    for k in range(model.k_freqs):
        print(
            f"{k+1:>2} | {freq_info['theoretical'][k]:>10.6f} | "
            f"{freq_info['learned_a'][k]:>9.6f} | {freq_info['error_a'][k]:>8.6f} | "
            f"{freq_info['learned_b'][k]:>9.6f} | {freq_info['error_b'][k]:>8.6f}"
        )

    print("mean error a:", freq_info["error_a"].mean())
    print("mean error b:", freq_info["error_b"].mean())
    print("SIFP-16 quantization error:", TWO_PI / PHASE_SCALE)
    return freq_info


def ablation_test(model: nn.Module, val_x, val_y, label: str = "model") -> dict:
    model.eval()
    results = {}
    with torch.no_grad():
        base_logits = model(val_x)
        base_acc = (base_logits.argmax(-1) == val_y).float().mean().item()
        results["baseline"] = base_acc

        if isinstance(model, PhaseAccumulatorNetwork):
            orig_w = model.phase_mix.weight.data.clone()
            model.phase_mix.weight.data.zero_()
            abl = model(val_x)
            results["zero_phase_mixing"] = (abl.argmax(-1) == val_y).float().mean().item()
            model.phase_mix.weight.data.copy_(orig_w)

            orig_fa = model.encoder_a.freq.data.clone()
            orig_fb = model.encoder_b.freq.data.clone()
            model.encoder_a.freq.data = torch.rand_like(orig_fa) * TWO_PI
            model.encoder_b.freq.data = torch.rand_like(orig_fb) * TWO_PI
            abl = model(val_x)
            results["randomize_frequencies"] = (abl.argmax(-1) == val_y).float().mean().item()
            model.encoder_a.freq.data.copy_(orig_fa)
            model.encoder_b.freq.data.copy_(orig_fb)

            orig_ref = model.phase_gate.ref_phase.data.clone()
            model.phase_gate.ref_phase.data.zero_()
            abl = model(val_x)
            results["zero_ref_phases"] = (abl.argmax(-1) == val_y).float().mean().item()
            model.phase_gate.ref_phase.data.copy_(orig_ref)

    print(f"ablation results [{label}]")
    for key, acc in results.items():
        print(f"  {key:<24} {acc:.4f}")
    return results


def plot_training_history(hist: TrainHistory, title: str):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(hist.steps, hist.train_loss, label="train")
    axes[0].plot(hist.steps, hist.val_loss, label="val")
    axes[0].set_title("loss")
    axes[0].set_xlabel("step")
    axes[0].set_ylabel("cross entropy")
    axes[0].set_yscale("log")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].plot(hist.steps, hist.val_acc, label="val acc")
    if hist.grok_step is not None:
        axes[1].axvline(hist.grok_step, linestyle="--", alpha=0.6, label=f"grok @ {hist.grok_step:,}")
    axes[1].set_title("validation accuracy")
    axes[1].set_xlabel("step")
    axes[1].set_ylabel("accuracy")
    axes[1].set_ylim(0, 1.05)
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def _sweep_sub_cfg(cfg: TrainConfig, seed: int, **overrides) -> TrainConfig:
    return TrainConfig(
        p=overrides.get("p", cfg.p),
        k_freqs=overrides.get("k_freqs", cfg.k_freqs),
        n_steps=overrides.get("n_steps", cfg.n_steps),
        seed=seed,
        lr=cfg.lr,
        batch_size=cfg.batch_size,
        weight_decay=overrides.get("weight_decay", cfg.weight_decay),
        diversity_weight=overrides.get("diversity_weight", cfg.diversity_weight),
        val_samples=cfg.val_samples,
        use_compile=False,
        early_stop=cfg.early_stop,
        log_every=cfg.log_every,
        output_dir=cfg.output_dir,
        dry_run=cfg.dry_run,
    )


def _detect_mode_collapse(model: PhaseAccumulatorNetwork) -> bool:
    W = model.phase_mix.weight.detach().cpu().numpy()
    dominant = [int(np.argmax(np.abs(row))) for row in W]
    return len(set(dominant)) == 1


## Experiment runners

These functions correspond to the modes in your script.


In [ ]:
def run_main_comparison(cfg: TrainConfig):
    train_x, train_y, val_x, val_y = make_modular_dataset(cfg.p, seed=cfg.seed)

    pan = PhaseAccumulatorNetwork(cfg.p, cfg.k_freqs).to(DEVICE)
    tf = TransformerBaseline(cfg.p, cfg.d_model, cfg.n_heads, cfg.d_mlp).to(DEVICE)

    print("PAN params:", pan.count_parameters())
    print("TF params:", tf.count_parameters())
    print("TF / PAN ratio:", tf.count_parameters() / pan.count_parameters())

    hist_pan = train(pan, cfg, train_x, train_y, val_x, val_y, label="PAN")
    hist_tf = train(tf, cfg, train_x, train_y, val_x, val_y, label="TF")

    print("PAN grok step:", hist_pan.grok_step)
    print("TF grok step:", hist_tf.grok_step)

    return {
        "pan": pan,
        "tf": tf,
        "hist_pan": hist_pan,
        "hist_tf": hist_tf,
        "train_x": train_x,
        "train_y": train_y,
        "val_x": val_x,
        "val_y": val_y,
    }


def run_parameter_sweep(cfg: TrainConfig):
    train_x, train_y, val_x, val_y = make_modular_dataset(cfg.p, seed=cfg.seed)
    results = {}

    for k in range(1, 16):
        grok_steps = []
        for seed in [42, 123, 456]:
            pan = PhaseAccumulatorNetwork(cfg.p, k_freqs=k).to(DEVICE)
            scfg = TrainConfig(
                p=cfg.p,
                n_steps=cfg.n_steps,
                k_freqs=k,
                seed=seed,
                lr=cfg.lr,
                batch_size=cfg.batch_size,
                weight_decay=cfg.weight_decay,
                diversity_weight=cfg.diversity_weight,
                val_samples=cfg.val_samples,
                use_compile=False,
                early_stop=cfg.early_stop,
                log_every=cfg.log_every,
                output_dir=cfg.output_dir,
                dry_run=cfg.dry_run,
            )
            hist = train(pan, scfg, train_x, train_y, val_x, val_y, label=f"PAN-K{k}-s{seed}")
            grok_steps.append(hist.grok_step)

        n_grokked = sum(1 for s in grok_steps if s is not None)
        mean_step = np.mean([s for s in grok_steps if s is not None]) if n_grokked > 0 else None
        params = PhaseAccumulatorNetwork(cfg.p, k).count_parameters()
        results[k] = {
            "grok_steps": grok_steps,
            "n_grokked": n_grokked,
            "mean_step": mean_step,
            "params": params,
        }
        print(f"K={k:2d}: grokked {n_grokked}/3 | mean_step={mean_step} | params={params}")

    return results


def run_multi_prime(cfg: TrainConfig):
    primes = [43, 67, 89, 113, 127]
    results = {}

    for p in primes:
        train_x, train_y, val_x, val_y = make_modular_dataset(p, seed=cfg.seed)
        pan = PhaseAccumulatorNetwork(p, k_freqs=cfg.k_freqs).to(DEVICE)
        pcfg = TrainConfig(
            p=p,
            n_steps=cfg.n_steps,
            k_freqs=cfg.k_freqs,
            seed=cfg.seed,
            lr=cfg.lr,
            batch_size=cfg.batch_size,
            weight_decay=cfg.weight_decay,
            diversity_weight=cfg.diversity_weight,
            val_samples=cfg.val_samples,
            use_compile=False,
            early_stop=cfg.early_stop,
            log_every=cfg.log_every,
            output_dir=cfg.output_dir,
            dry_run=cfg.dry_run,
        )
        hist = train(pan, pcfg, train_x, train_y, val_x, val_y, label=f"PAN-P{p}")
        results[p] = {
            "grok_step": hist.grok_step,
            "final_acc": hist.val_acc[-1] if hist.val_acc else 0.0,
            "params": pan.count_parameters(),
        }
        print(f"P={p}: grok={results[p]['grok_step']} final_acc={results[p]['final_acc']:.4f} params={results[p]['params']}")
    return results


def run_tier3(cfg: TrainConfig):
    train_x, train_y, val_x, val_y = make_modular_dataset(cfg.p, seed=cfg.seed)
    t3cfg = TrainConfig(
        p=cfg.p,
        k_freqs=cfg.k_freqs,
        n_steps=cfg.n_steps,
        seed=cfg.seed,
        lr=cfg.lr,
        batch_size=cfg.batch_size,
        weight_decay=cfg.weight_decay,
        diversity_weight=cfg.diversity_weight,
        val_samples=cfg.val_samples,
        use_compile=False,
        early_stop=False,
        log_every=cfg.log_every,
        output_dir=cfg.output_dir,
        dry_run=cfg.dry_run,
    )
    pan = PhaseAccumulatorNetwork(cfg.p, cfg.k_freqs).to(DEVICE)
    hist = train(pan, t3cfg, train_x, train_y, val_x, val_y, label="PAN-T3", record_checkpoints=True)
    return {"pan": pan, "hist": hist, "train_x": train_x, "train_y": train_y, "val_x": val_x, "val_y": val_y}


def run_dw_sweep(cfg: TrainConfig):
    dw_values = [0.0, 0.005, 0.01, 0.02, 0.05, 0.1]
    seeds = [42, 123, 456, 789, 999]
    train_x, train_y, val_x, val_y = make_modular_dataset(cfg.p, seed=cfg.seed)
    results = {}

    for dw in dw_values:
        grok_steps, final_accs, collapses = [], [], []
        for seed in seeds:
            pan = PhaseAccumulatorNetwork(cfg.p, 9).to(DEVICE)
            scfg = _sweep_sub_cfg(cfg, seed, k_freqs=9, weight_decay=0.01, diversity_weight=dw)
            hist = train(pan, scfg, train_x, train_y, val_x, val_y, label=f"DW{dw:.3f}-s{seed}")
            grok_steps.append(hist.grok_step)
            final_accs.append(hist.val_acc[-1] if hist.val_acc else 0.0)
            collapses.append(_detect_mode_collapse(pan))

        results[dw] = {
            "grok_steps": grok_steps,
            "n_grokked": sum(1 for s in grok_steps if s is not None),
            "mean_step": np.mean([s for s in grok_steps if s is not None]) if any(s is not None for s in grok_steps) else None,
            "mean_acc": float(np.mean(final_accs)),
            "n_collapse": int(sum(collapses)),
            "final_accs": final_accs,
        }
        print(dw, results[dw])
    return results


def run_wd_sweep(cfg: TrainConfig):
    wd_values = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1]
    seeds = [42, 123, 456]
    train_x, train_y, val_x, val_y = make_modular_dataset(cfg.p, seed=cfg.seed)
    results = {}

    for wd in wd_values:
        grok_steps, final_accs = [], []
        for seed in seeds:
            pan = PhaseAccumulatorNetwork(cfg.p, 9).to(DEVICE)
            scfg = _sweep_sub_cfg(cfg, seed, k_freqs=9, weight_decay=wd)
            hist = train(pan, scfg, train_x, train_y, val_x, val_y, label=f"WD{wd:.3f}-s{seed}")
            grok_steps.append(hist.grok_step)
            final_accs.append(hist.val_acc[-1] if hist.val_acc else 0.0)

        results[wd] = {
            "grok_steps": grok_steps,
            "n_grokked": sum(1 for s in grok_steps if s is not None),
            "mean_step": np.mean([s for s in grok_steps if s is not None]) if any(s is not None for s in grok_steps) else None,
            "mean_acc": float(np.mean(final_accs)),
        }
        print(wd, results[wd])
    return results


def run_k8_sweep(cfg: TrainConfig):
    seeds = [42, 123, 456, 789, 999, 1234, 2345, 3456, 4567, 5678]
    train_x, train_y, val_x, val_y = make_modular_dataset(cfg.p, seed=cfg.seed)
    grok_steps, final_accs, peak_accs = [], [], []

    for seed in seeds:
        pan = PhaseAccumulatorNetwork(cfg.p, 8).to(DEVICE)
        scfg = _sweep_sub_cfg(cfg, seed, k_freqs=8, weight_decay=0.01)
        hist = train(pan, scfg, train_x, train_y, val_x, val_y, label=f"K8-s{seed}")
        grok_steps.append(hist.grok_step)
        final_accs.append(hist.val_acc[-1] if hist.val_acc else 0.0)
        peak_accs.append(max(hist.val_acc) if hist.val_acc else 0.0)
        print(f"seed={seed}: grok={hist.grok_step} peak={peak_accs[-1]:.4f} final={final_accs[-1]:.4f}")

    return {
        "grok_steps": grok_steps,
        "n_grokked": sum(1 for s in grok_steps if s is not None),
        "peak_accs": peak_accs,
        "final_accs": final_accs,
    }


def run_tf_sweep(cfg: TrainConfig):
    d_models = [8, 16, 32, 64, 128]
    seeds = [42, 123, 456]
    train_x, train_y, val_x, val_y = make_modular_dataset(cfg.p, seed=cfg.seed)
    results = {}

    for d in d_models:
        grok_steps, final_accs = [], []
        n_heads = max(1, d // 16)
        d_mlp = 4 * d
        for seed in seeds:
            tf = TransformerBaseline(cfg.p, d_model=d, n_heads=n_heads, d_mlp=d_mlp).to(DEVICE)
            scfg = _sweep_sub_cfg(cfg, seed, weight_decay=1.0)
            hist = train(tf, scfg, train_x, train_y, val_x, val_y, label=f"TF-d{d}-s{seed}")
            grok_steps.append(hist.grok_step)
            final_accs.append(hist.val_acc[-1] if hist.val_acc else 0.0)

        params = TransformerBaseline(cfg.p, d_model=d, n_heads=n_heads, d_mlp=d_mlp).count_parameters()
        results[d] = {
            "grok_steps": grok_steps,
            "n_grokked": sum(1 for s in grok_steps if s is not None),
            "mean_step": np.mean([s for s in grok_steps if s is not None]) if any(s is not None for s in grok_steps) else None,
            "params": params,
            "n_heads": n_heads,
            "d_mlp": d_mlp,
            "mean_acc": float(np.mean(final_accs)),
        }
        print(f"d={d}: {results[d]}")
    return results


def run_held_out_primes(cfg: TrainConfig):
    primes = [59, 71, 97]
    results = {}
    for p in primes:
        train_x, train_y, val_x, val_y = make_modular_dataset(p, seed=cfg.seed)
        pan = PhaseAccumulatorNetwork(p, 9).to(DEVICE)
        pcfg = _sweep_sub_cfg(cfg, cfg.seed, p=p, k_freqs=9, weight_decay=0.01)
        hist = train(pan, pcfg, train_x, train_y, val_x, val_y, label=f"PAN-P{p}-held")
        results[p] = {
            "grok_step": hist.grok_step,
            "final_acc": hist.val_acc[-1] if hist.val_acc else 0.0,
            "params": pan.count_parameters(),
        }
        print(f"P={p}: {results[p]}")
    return results


## Shared base configs

The point here is to make the experiment assumptions explicit.

The notebook keeps the same general defaults as your script, but for PAN-heavy experiments you will usually want:

- `weight_decay=0.01`
- `diversity_weight=0.01`
- `use_compile=False` inside sweeps and tier 3


In [ ]:
BASE_COMPARE_CFG = TrainConfig(
    p=113,
    k_freqs=5,
    n_steps=50_000,
    batch_size=256,
    lr=1e-3,
    weight_decay=1.0,      # matches current script default
    diversity_weight=0.01,
    log_every=200,
    seed=42,
    val_samples=1024,
    use_compile=False,
    early_stop=True,
)

BASE_PAN_CFG = TrainConfig(
    p=113,
    k_freqs=9,
    n_steps=100_000,
    batch_size=256,
    lr=1e-3,
    weight_decay=0.01,
    diversity_weight=0.01,
    log_every=200,
    seed=42,
    val_samples=1024,
    use_compile=False,
    early_stop=True,
)

BASE_TIER3_CFG = TrainConfig(
    p=113,
    k_freqs=9,
    n_steps=100_000,
    batch_size=256,
    lr=1e-3,
    weight_decay=0.01,
    diversity_weight=0.01,
    log_every=200,
    seed=42,
    val_samples=1024,
    use_compile=False,
    early_stop=False,
)

print(BASE_COMPARE_CFG)
print(BASE_PAN_CFG)
print(BASE_TIER3_CFG)


## Compare mode

This reproduces the main head-to-head run between PAN and the transformer.


In [ ]:
RUN_COMPARE = False

if RUN_COMPARE:
    compare_results = run_main_comparison(BASE_COMPARE_CFG)
    plot_training_history(compare_results["hist_pan"], "PAN compare run")
    plot_training_history(compare_results["hist_tf"], "Transformer compare run")
    analyze_pan(compare_results["pan"])
    ablation_test(compare_results["pan"], compare_results["val_x"], compare_results["val_y"], label="PAN")


## Sweep mode

This reproduces the `K=1..15` parameter sweep. It is the main way to test the README claims about minimum reliable `K`.


In [ ]:
RUN_K_SWEEP = False

if RUN_K_SWEEP:
    k_sweep_results = run_parameter_sweep(BASE_PAN_CFG)
    k_summary = {
        k: {
            "n_grokked": v["n_grokked"],
            "mean_step": v["mean_step"],
            "params": v["params"],
        }
        for k, v in k_sweep_results.items()
    }
    k_summary


## Primes mode

This checks whether the same PAN configuration works across multiple primes with the same hyperparameters.


In [ ]:
RUN_PRIMES = False

if RUN_PRIMES:
    prime_results = run_multi_prime(BASE_PAN_CFG)
    prime_results


## Tier 3 mode

This is the mechanistic run. It records checkpoints so you can inspect:

- frequency trajectories,
- Fourier concentration over time,
- grokking vs lock-in timing.


In [ ]:
def plot_tier3(hist: TrainHistory, pan: PhaseAccumulatorNetwork, p: int):
    if not hist.freq_checkpoints:
        print("no checkpoint data recorded")
        return

    ck_steps = sorted(hist.freq_checkpoints.keys())
    theoretical = np.array([(k + 1) * TWO_PI / p for k in range(pan.k_freqs)])
    colors = plt.cm.tab10(np.linspace(0, 0.9, pan.k_freqs))

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    ax1 = axes[0, 0]
    for k in range(pan.k_freqs):
        freqs = [hist.freq_checkpoints[s]["learned_a"][k] for s in ck_steps]
        ax1.plot(ck_steps, freqs, color=colors[k], lw=1.5, label=f"k={k+1}")
        ax1.axhline(theoretical[k], color=colors[k], lw=0.8, ls="--", alpha=0.5)
    if hist.grok_step:
        ax1.axvline(hist.grok_step, color="black", ls=":", alpha=0.7)
    ax1.set_title("encoder A trajectory")
    ax1.set_xlabel("step")
    ax1.set_ylabel("freq")
    ax1.grid(alpha=0.25)

    ax2 = axes[0, 1]
    for k in range(pan.k_freqs):
        freqs = [hist.freq_checkpoints[s]["learned_b"][k] for s in ck_steps]
        ax2.plot(ck_steps, freqs, color=colors[k], lw=1.5, label=f"k={k+1}")
        ax2.axhline(theoretical[k], color=colors[k], lw=0.8, ls="--", alpha=0.5)
    if hist.grok_step:
        ax2.axvline(hist.grok_step, color="black", ls=":", alpha=0.7)
    ax2.set_title("encoder B trajectory")
    ax2.set_xlabel("step")
    ax2.set_ylabel("freq")
    ax2.grid(alpha=0.25)

    ax3 = axes[1, 0]
    ax3.plot(hist.fourier_conc_steps, hist.fourier_conc_values, lw=2)
    if hist.grok_step:
        ax3.axvline(hist.grok_step, color="black", ls=":", alpha=0.7)
    ax3.set_title("decoder Fourier concentration")
    ax3.set_xlabel("step")
    ax3.set_ylabel("concentration")
    ax3.set_ylim(0, 1)
    ax3.grid(alpha=0.25)

    ax4 = axes[1, 1]
    final_info = hist.freq_checkpoints[ck_steps[-1]]
    x_pos = np.arange(pan.k_freqs)
    width = 0.35
    ax4.bar(x_pos - width / 2, final_info["error_a"], width, label="encoder A")
    ax4.bar(x_pos + width / 2, final_info["error_b"], width, label="encoder B")
    ax4.axhline(TWO_PI / PHASE_SCALE, color="red", ls="--", alpha=0.7, label="SIFP-16 quant")
    ax4.set_xticks(x_pos)
    ax4.set_xticklabels([f"k={k+1}" for k in range(pan.k_freqs)])
    ax4.set_title("final angular error")
    ax4.set_xlabel("frequency")
    ax4.set_ylabel("error")
    ax4.grid(alpha=0.25)
    ax4.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
RUN_TIER3 = False

if RUN_TIER3:
    tier3_results = run_tier3(BASE_TIER3_CFG)
    plot_training_history(tier3_results["hist"], "PAN tier3 run")
    plot_tier3(tier3_results["hist"], tier3_results["pan"], BASE_TIER3_CFG.p)
    analyze_pan(tier3_results["pan"])
    ablation_test(tier3_results["pan"], tier3_results["val_x"], tier3_results["val_y"], label="PAN-T3")


## Diversity-weight sweep


In [ ]:
RUN_DW_SWEEP = False

if RUN_DW_SWEEP:
    dw_results = run_dw_sweep(BASE_PAN_CFG)
    dw_results


## Weight-decay sweep


In [ ]:
RUN_WD_SWEEP = False

if RUN_WD_SWEEP:
    wd_results = run_wd_sweep(BASE_PAN_CFG)
    wd_results


## K=8 anomaly sweep


In [ ]:
RUN_K8_SWEEP = False

if RUN_K8_SWEEP:
    k8_cfg = TrainConfig(**{**BASE_PAN_CFG.__dict__, "n_steps": 200_000, "k_freqs": 8})
    k8_results = run_k8_sweep(k8_cfg)
    k8_results


## Transformer minimum-size sweep


In [ ]:
RUN_TF_SWEEP = False

if RUN_TF_SWEEP:
    tf_cfg = TrainConfig(**{**BASE_PAN_CFG.__dict__, "n_steps": 100_000, "weight_decay": 1.0})
    tf_results = run_tf_sweep(tf_cfg)
    tf_results


## Held-out primes


In [ ]:
RUN_HELD_OUT_PRIMES = False

if RUN_HELD_OUT_PRIMES:
    held_cfg = TrainConfig(**{**BASE_PAN_CFG.__dict__, "n_steps": 200_000, "k_freqs": 9})
    held_results = run_held_out_primes(held_cfg)
    held_results


## Optional: compact claim checks

These are not absolute proofs. They are convenience checks against the qualitative claims in the README.


In [ ]:
def claim_check_compare(results):
    hp = results["hist_pan"]
    ht = results["hist_tf"]
    pan = results["pan"]
    tf = results["tf"]

    print("claim: parameter gap is large")
    print("  ratio:", tf.count_parameters() / pan.count_parameters())

    print("claim: PAN can exceed 99% val accuracy")
    print("  max PAN val acc:", max(hp.val_acc) if hp.val_acc else None)

    print("claim: transformer can also solve the task")
    print("  max TF val acc:", max(ht.val_acc) if ht.val_acc else None)


def claim_check_k_sweep(results):
    reliable = [k for k, r in results.items() if r["n_grokked"] >= 2]
    print("reliable K values:", reliable)
    print("minimum reliable K:", min(reliable) if reliable else None)


def claim_check_primes(results):
    print("grok count:", sum(1 for r in results.values() if r["grok_step"] is not None), "/", len(results))
    print("final accs:", {p: round(r["final_acc"], 4) for p, r in results.items()})


## Notes on reproducibility

A few results in this project are intentionally sensitive to configuration:

- compile mode can change trajectories,
- MPS can differ from CPU/CUDA in accumulation behavior,
- the PAN is much more sensitive to weight decay than the transformer,
- small `K` values are seed-sensitive and sometimes bifurcate sharply.

So when a result is near a threshold, the notebook should be used to inspect the full trajectory rather than relying only on one scalar summary.
